### Funkcja Tranform - Zadania

In [2]:
# Split Apply Combine

>> Zadanie 1: Normalizacja wyników

Masz DataFrame z wynikami uczniów z różnych klas. 

Znormalizuj wyniki w obrębie każdej klasy (z-score).

z-score = (x - mean) / std

z-score pokazuje, ile odchyleń standardowych dany wynik jest od średniej swojej klasy.

In [3]:
import pandas as pd

df = pd.DataFrame({
    'klasa': ['A','A','A','B','B','B','C','C','C'],
    'uczen': ['Anna','Bartek','Celina','Dawid','Ewa','Filip','Gosia','Henryk','Iga'],
    'wynik': [85, 90, 78, 70, 65, 80, 95, 88, 92],
})

In [4]:
print(type(print('x')))

display(df)

x
<class 'NoneType'>


,klasa,uczen,wynik
0,A,Anna,85
1,A,Bartek,90
2,A,Celina,78
3,B,Dawid,70
4,B,Ewa,65
5,B,Filip,80
6,C,Gosia,95
7,C,Henryk,88
8,C,Iga,92


`.transform( lambda x: fn(x) )` = `.tranform(fn)`

In [5]:
def z_score(x):
    return (x - x.mean()) / x.std()

df['wynik_znrom'] = (
    # df.groupby('klasa')['wynik'].transform( z_score )
    # df.groupby('klasa')['wynik'].transform( lambda x: z_score(x) )
    # df.groupby('klasa')['wynik'].transform( lambda x: None or z_score(x) )
    df.groupby('klasa')['wynik'] # grupuje po klasie i wybiera z każdej "klasy" kolumnę wynik
        .transform( lambda x: print(type(x)) or z_score(x) )
)

def my_mean(x):
    return x.mean()

# df['mean_per_klasa'] = df.groupby('klasa')['wynik'].transform(my_mean) # własna funkcja agregująca
# df['mean_per_klasa'] = df.groupby('klasa')['wynik'].transform(lambda x: x.mean()) # wbudowana fn agregująca
df['mean_per_klasa'] = df.groupby('klasa')['wynik'].transform('mean') # wbudowana fn agregująca - najlepsza!

print(df)

<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
  klasa   uczen  wynik  wynik_znrom  mean_per_klasa
0     A    Anna     85     0.110600       84.333333
1     A  Bartek     90     0.940102       84.333333
2     A  Celina     78    -1.050702       84.333333
3     B   Dawid     70    -0.218218       71.666667
4     B     Ewa     65    -0.872872       71.666667
5     B   Filip     80     1.091089       71.666667
6     C   Gosia     95     0.949158       91.666667
7     C  Henryk     88    -1.044074       91.666667
8     C     Iga     92     0.094916       91.666667



---
### Zadanie 2: Ranking wewnątrz Grupy

Masz dane o sprzedawcach w różnych działach. Oblicz: 

(1) ranking sprzedawcy w jego dziale (1 = najlepszy), 

(2) udział procentowy danego sprzedawcy w sprzedaży działu.

In [ ]:
import pandas as pd
df = pd.DataFrame({
    'dzial': ['Tech','Tech', 'Tech', 'Tech','Tech','HR','HR','Marketing','Marketing','Marketing'],
    'sprzedawca': ['Adam2', 'Adam3', 'Adam','Basia','Cezary','Diana','Emil','Franek','Grażyna','Hubert'], # ranking dla każdego sprzedawcy wymaga tego samego kształtu ramki -> transform
    'sprzedaz': [501, 501, 100, 800, 300, 400, 600, 200, 350, 450]
})

Utwórz nowe kolumny: 'ranking', 'udzial_proc'

In [ ]:
# podpowiedź do komórki poniżej

x = pd.Series([0, 1, 2])  # x jest kolumną (wektorem wartości)

y = 2                     # y jest skalarem (pojedynczą wartością)

z = y * x                 # z jest wynikiem operacji wektorowej:
                          # skalar y zostaje pomnożony przez każdy
                          # element wektora x

# udział każdego x_i w całej kolumnie gdzie x = [x_1, x_2, ...] a x.sum() zwraca sumę wartości x
mean_ = x / x.sum() 

# każdy element wektora x jest mnożony przez sumę jego elementów
# x = [0,  / (0 + 1 + 2)
#      1,  / (0 + 1 + 2)
#      2]  / (0 + 1 + 2)]

print(z)

0    0
1    2
2    4
dtype: int64


In [ ]:
# ------------------
# Ranking
# ------------------
df['ranking'] = df.groupby('dzial')['sprzedaz'].transform( # dzielimy na grupy 'dzial' -> przekazujemy kolumnę 'sprzedaz' każdej grupy do funkcji transform
    
    # --- wersja z method='min'
    # lambda x: x.rank(ascending=False, method='min') # remisy - obaj dostają tę samą rangę, 
                                                      # ale kolejne liczby porządkowe rankingu "znikają" 
                                                      # kosztem remisu, np: 1, 2, 2, 2, 5

    lambda x: x.rank(ascending=False, method='dense') # 'dense' zachowuje kolejność rang nieprzerwaną: 1, 2, 2 ,2 ,3 etc.
)

def udzial_proc(x):
    print(type(x)) # debug print
    return ((x / x.sum()) * 100).round(1)  

# ------------------
# Udział procentowy
# ------------------
df['udzial_proc'] = df.groupby('dzial')['sprzedaz'].transform( 
    udzial_proc
)

display(df)

<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>


,dzial,sprzedawca,sprzedaz,ranking,udzial_proc
0,Tech,Adam2,501,2.0,22.8
1,Tech,Adam3,501,2.0,22.8
2,Tech,Adam,100,4.0,4.5
3,Tech,Basia,800,1.0,36.3
4,Tech,Cezary,300,3.0,13.6
5,HR,Diana,400,2.0,40.0
6,HR,Emil,600,1.0,60.0
7,Marketing,Franek,200,3.0,20.0
8,Marketing,Grażyna,350,2.0,35.0
9,Marketing,Hubert,450,1.0,45.0


In [ ]:
df = df.assign(
    ranking=df.groupby('dzial')['sprzedaz'].transform(lambda x: x.rank(ascending=False, method='dense')),
    udzial_proc=df.groupby('dzial')['sprzedaz'].transform(lambda x: ((x / x.sum()) * 100).round(1) )
).sort_values(['dzial', 'ranking'])     # sortujemy najpierw po kolumnie 'dzial', następnie po 'ranking'

display(df)

,dzial,sprzedawca,sprzedaz,ranking,udzial_proc
6,HR,Emil,600,1.0,60.0
5,HR,Diana,400,2.0,40.0
9,Marketing,Hubert,450,1.0,45.0
8,Marketing,Grażyna,350,2.0,35.0
7,Marketing,Franek,200,3.0,20.0
3,Tech,Basia,800,1.0,36.3
0,Tech,Adam2,501,2.0,22.8
1,Tech,Adam3,501,2.0,22.8
4,Tech,Cezary,300,3.0,13.6
2,Tech,Adam,100,4.0,4.5


In [ ]:
# Notatka odnośnie funkcji assign:

df_ = pd.DataFrame({
    'x': [0, 1],
    'y': [1, 2]
})

display(df_)

# tworzymy nowe kolumny
df_ = df_.assign(nowa_kolumna=pd.Series([3, 4]), z=pd.Series([5, 6]))

df_

,x,y
0,0,1
1,1,2


,x,y,nowa_kolumna,z
0,0,1,3,5
1,1,2,4,6
